In [ ]:
# install dependencies
#!pip install -q transformers wandb scikit-learn tqdm
'''This notebook was used to train and finetune the full AST model on Google Colab.'''

In [3]:
# mount drive and extract cache
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
import os
import zipfile

DRIVE_ROOT = "/content/drive/MyDrive/dlgenai"
MODEL_DIR  = os.path.join(DRIVE_ROOT, "models")
os.makedirs(MODEL_DIR, exist_ok=True)

CACHE_LOCAL = "/content/ast_cache"

if not os.path.exists(CACHE_LOCAL):
    print("Extracting ast_cache...")
    with zipfile.ZipFile(os.path.join(DRIVE_ROOT, "ast_cache.zip"), "r") as zf:
        zf.extractall("/content")
    print("Done.")

# confirm structure
train_files = [f for f in os.listdir(os.path.join(CACHE_LOCAL, "train")) if f.endswith(".pt")]
val_files = [f for f in os.listdir(os.path.join(CACHE_LOCAL, "val"))   if f.endswith(".pt")]
print(f"Train files: {len(train_files)} | Val files: {len(val_files)}")

Extracting ast_cache...
Done.
Train files: 1600 | Val files: 400


In [5]:
# Necessary libraries
import glob
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import f1_score, accuracy_score
from transformers import ASTForAudioClassification
import wandb
from tqdm import tqdm
import warnings
warnings.filterwarnings("ignore")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

Device: cuda


In [ ]:
# Configs
GENRES = ["blues", "classical", "country", "disco", "hiphop",
                "jazz", "metal", "pop", "reggae", "rock"]
GENRE_TO_IDX = {g: i for i, g in enumerate(GENRES)}
IDX_TO_GENRE = {i: g for i, g in enumerate(GENRES)}

AST_MODEL_ID = "MIT/ast-finetuned-audioset-10-10-0.4593"

BATCH_SIZE = 16
LR = 1e-5
WEIGHT_DECAY = 0.01
NUM_EPOCHS = 10

In [ ]:
class CachedASTDataset(Dataset):
    """Loads precomputed AST feature tensors from disk."""
    def __init__(self, cache_dir, split):
        split_dir = os.path.join(cache_dir, split)
        self.files = sorted(glob.glob(os.path.join(split_dir, "*.pt")))

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        item = torch.load(self.files[idx], weights_only=False)
        # cache files store tensors under key "input_values" and integer label
        return item["input_values"], item["label"]

In [ ]:
train_dataset = CachedASTDataset(CACHE_LOCAL, "train")
val_dataset = CachedASTDataset(CACHE_LOCAL, "val")

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=2, pin_memory=True)
val_loader = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=2, pin_memory=True)

print(f"Train: {len(train_dataset)} | Val: {len(val_dataset)}")

Train: 1600 | Val: 400


In [17]:
sample_x, sample_y = train_dataset[0]
print(f"Input shape: {sample_x.shape} | Label: {sample_y} ({IDX_TO_GENRE[sample_y]})")

Input shape: torch.Size([1024, 128]) | Label: 2 (country)


In [ ]:
# AST Model — full fine-tuning, no layers frozen
model = ASTForAudioClassification.from_pretrained(
    AST_MODEL_ID, num_labels=10, ignore_mismatched_sizes=True
)
model = model.to(DEVICE)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total parameters:     {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/346M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/203 [00:00<?, ?it/s]

ASTForAudioClassification LOAD REPORT from: MIT/ast-finetuned-audioset-10-10-0.4593
Key                     | Status   |                                                                                        
------------------------+----------+----------------------------------------------------------------------------------------
classifier.dense.weight | MISMATCH | Reinit due to size mismatch ckpt: torch.Size([527, 768]) vs model:torch.Size([10, 768])
classifier.dense.bias   | MISMATCH | Reinit due to size mismatch ckpt: torch.Size([527]) vs model:torch.Size([10])          

Notes:
- MISMATCH	:ckpt weights were loaded, but they did not match the original empty weight shapes.


Total parameters:     86,196,490
Trainable parameters: 86,196,490


In [ ]:
# training utilities
criterion = nn.CrossEntropyLoss()

def run_epoch(model, loader, criterion, optimizer, training, device, desc=""):
    model.train() if training else model.eval()
    total_loss, preds, labels_all = 0.0, [], []

    ctx = torch.enable_grad() if training else torch.no_grad()
    with ctx:
        for inputs, labels in tqdm(loader, desc=desc):
            inputs = inputs.to(device)
            labels = labels.to(device)
            if training:
                optimizer.zero_grad()
            logits = model(inputs).logits
            loss = criterion(logits, labels)
            if training:
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                optimizer.step()
            total_loss += loss.item() * inputs.size(0)
            preds.extend(logits.argmax(dim=1).detach().cpu().numpy())
            labels_all.extend(labels.detach().cpu().numpy())

    avg_loss = total_loss / len(loader.dataset)
    f1 = f1_score(labels_all, preds, average="macro")
    acc = accuracy_score(labels_all, preds)
    return avg_loss, f1, acc

In [20]:
# WandB login
wandb.login()

wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:

 2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.
wandb: Paste your API key and hit enter:

 ··········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: W&B API key is configured. Use `wandb login --relogin` to force relogin


True

In [21]:
optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS)

In [22]:
wandb.init(
    project="21f3002298-dl-genai-project",
    name="ast-src-full-finetune",
    config={
        "model":             AST_MODEL_ID,
        "epochs":            NUM_EPOCHS,
        "lr":                LR,
        "weight_decay":      WEIGHT_DECAY,
        "batch_size":        BATCH_SIZE,
        "frozen_layers":     "none (full fine-tune)",
        "samples_per_genre": 200,
    }
)

In [23]:
best_val_f1 = 0.0

for epoch in range(NUM_EPOCHS):
    train_loss, train_f1, train_acc = run_epoch(
        model, train_loader, criterion, optimizer, training=True,
        device=DEVICE, desc=f"Epoch {epoch+1}/{NUM_EPOCHS} [Train]"
    )
    val_loss, val_f1, val_acc = run_epoch(
        model, val_loader, criterion, optimizer, training=False,
        device=DEVICE, desc=f"Epoch {epoch+1}/{NUM_EPOCHS} [Val]"
    )
    scheduler.step()
    torch.cuda.empty_cache()

    wandb.log({
        "epoch":      epoch + 1,
        "train_loss": train_loss, "train_f1": train_f1, "train_acc": train_acc,
        "val_loss":   val_loss,   "val_f1":   val_f1,   "val_acc":   val_acc,
        "lr":         scheduler.get_last_lr()[0],
    })
    print(f"Epoch {epoch+1}/{NUM_EPOCHS}: train_f1={train_f1:.4f} | val_f1={val_f1:.4f}")

    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        torch.save(model.state_dict(),
                   os.path.join(MODEL_DIR, "ast_src_full_finetune_best.pth"))
        print(f"  Saved best model (val_f1={val_f1:.4f})")

Epoch 1/10 [Val]: 100%|██████████| 25/25 [00:35<00:00,  1.44s/it]


Epoch 1/10: train_f1=0.5472 | val_f1=0.7838
  Saved best model (val_f1=0.7838)


Epoch 2/10 [Val]: 100%|██████████| 25/25 [00:36<00:00,  1.46s/it]


Epoch 2/10: train_f1=0.8586 | val_f1=0.8412
  Saved best model (val_f1=0.8412)


Epoch 3/10 [Val]: 100%|██████████| 25/25 [00:36<00:00,  1.46s/it]


Epoch 3/10: train_f1=0.9541 | val_f1=0.8250


... (10 outputs left out)

Epoch 9/10 [Val]: 100%|██████████| 25/25 [00:36<00:00,  1.45s/it]


Epoch 9/10: train_f1=1.0000 | val_f1=0.8592


Epoch 10/10 [Val]: 100%|██████████| 25/25 [00:35<00:00,  1.44s/it]

Epoch 10/10: train_f1=1.0000 | val_f1=0.8592


In [24]:
# Finish wandb run
wandb.finish()

epoch,▁▂▃▃▄▅▆▆▇█
lr,█▇▇▆▅▃▂▂▁▁
train_acc,▁▆▇███████
train_f1,▁▆▇███████
train_loss,█▃▂▁▁▁▁▁▁▁
val_acc,▁▆▅▇█▆▇███
val_f1,▁▆▅▇█▆▇███
val_loss,█▂▂▁▁▃▃▃▃▃
epoch,10
lr,0
train_acc,1


## AST Full Fine-tune with 400 samples

In [ ]:
# extract genres_stems and ESC-50 if not already present
import os
import zipfile

DRIVE_ROOT = "/content/drive/MyDrive/dlgenai"
GENRES_STEMS_DIR = "/content/genres_stems"
ESC50_AUDIO_DIR = "/content/ESC-50-master/audio"

if not os.path.exists(GENRES_STEMS_DIR):
    print("Extracting genres_stems...")
    with zipfile.ZipFile(os.path.join(DRIVE_ROOT, "genres_stems.zip"), "r") as zf:
        zf.extractall("/content")
    print("Done.")

if not os.path.exists(ESC50_AUDIO_DIR):
    print("Downloading ESC-50...")
    os.system("wget -q https://github.com/karolpiczak/ESC-50/archive/master.zip -O /content/ESC-50.zip")
    with zipfile.ZipFile("/content/ESC-50.zip", "r") as zf:
        zf.extractall("/content")
    print("Done.")

Extracting genres_stems...
Done.
Done.


In [ ]:
# precompute 400 samples/genre into a fresh cache
import random
import numpy as np
import torch
from sklearn.model_selection import train_test_split
from transformers import AutoFeatureExtractor
from tqdm import tqdm
import librosa

GENRES = ["blues", "classical", "country", "disco", "hiphop",
                "jazz", "metal", "pop", "reggae", "rock"]
GENRE_TO_IDX = {g: i for i, g in enumerate(GENRES)}

AST_MODEL_ID = "MIT/ast-finetuned-audioset-10-10-0.4593"
SR = 16000
DURATION = 10.0
TARGET_LEN = int(SR * DURATION)
SAMPLES_PER_GENRE = 400
VAL_SPLIT = 0.2
DATA_SEED_400 = 99      # different seed so mixes are independent of the 200-sample cache
STEMS = ["drums", "vocals", "bass", "other"]
GAIN_MIN = 0.5
GAIN_MAX = 1.5
SNR_MIN_DB = 5.0
SNR_MAX_DB = 20.0
CACHE_400 = "/content/ast_cache_400"

In [ ]:
# Helper functions
def _pad_or_trim(y, length):
    if len(y) >= length:
        return y[:length]
    return np.pad(y, (0, length - len(y)))


def make_augmented_mix(genre, data_root, sr, duration, rng):
    """
    Builds one training sample via cross-song stem mixing.
    Each stem is drawn from an independently chosen song of the same genre.
    """
    target_len = int(sr * duration)
    genre_dir = os.path.join(data_root, genre)
    songs = sorted(os.listdir(genre_dir))
    mix = np.zeros(target_len, dtype=np.float32)

    for stem in STEMS:
        song = rng.choice(songs)
        path = os.path.join(genre_dir, song, f"{stem}.wav")
        try:
            y, _ = librosa.load(path, sr=sr)
            if len(y) > target_len:
                start = rng.randint(0, len(y) - target_len)
                y = y[start: start + target_len]
            else:
                y = _pad_or_trim(y, target_len)
            mix += y * rng.uniform(GAIN_MIN, GAIN_MAX)
        except Exception:
            pass

    noise_files = [f for f in os.listdir(ESC50_AUDIO_DIR) if f.endswith(".wav")]
    try:
        noise, _ = librosa.load(
            os.path.join(ESC50_AUDIO_DIR, rng.choice(noise_files)), sr=sr
        )
        noise = _pad_or_trim(noise, target_len)
        snr_db = rng.uniform(SNR_MIN_DB, SNR_MAX_DB)
        sig_rms = np.sqrt(np.mean(mix   ** 2)) + 1e-9
        noise_rms = np.sqrt(np.mean(noise ** 2)) + 1e-9
        target_rms = sig_rms / (10 ** (snr_db / 20.0))
        mix = mix + noise * (target_rms / noise_rms)
    except Exception:
        pass

    peak = np.max(np.abs(mix))
    if peak > 0:
        mix /= (peak + 1e-9)
    return mix

In [28]:
def precompute_ast_features_400(data_root, cache_dir, genres, sr, duration,
                                 samples_per_genre, val_split, seed):
    """
    Generates augmented mixes, runs the AST feature extractor, and saves
    each result as a .pt file. Skips files that already exist on disk.
    """
    feature_extractor = AutoFeatureExtractor.from_pretrained(AST_MODEL_ID)

    all_items = [(g, i) for g in genres for i in range(samples_per_genre)]
    train_items, val_items = train_test_split(
        all_items, test_size=val_split, random_state=seed,
        stratify=[g for g, _ in all_items]
    )

    for split, items in [("train", train_items), ("val", val_items)]:
        split_dir = os.path.join(cache_dir, split)
        os.makedirs(split_dir, exist_ok=True)
        for pos, (genre, idx) in enumerate(tqdm(items, desc=f"Precomputing [{split}]")):
            out_path = os.path.join(split_dir, f"{split}_{pos:05d}.pt")
            if os.path.exists(out_path):
                continue
            rng = random.Random(seed + pos)
            mix = make_augmented_mix(genre, data_root, sr, duration, rng)
            inputs = feature_extractor(
                mix, sampling_rate=sr, return_tensors="pt", padding=True
            )
            torch.save({
                "input_values": inputs["input_values"].squeeze(0),
                "label":        GENRE_TO_IDX[genre]
            }, out_path)

    print("Precomputation complete.")

In [29]:
# Precomputing features with 400 samples per genre
precompute_ast_features_400(
    data_root=GENRES_STEMS_DIR, cache_dir=CACHE_400, genres=GENRES,
    sr=SR, duration=DURATION, samples_per_genre=SAMPLES_PER_GENRE,
    val_split=VAL_SPLIT, seed=DATA_SEED_400
)

preprocessor_config.json:   0%|          | 0.00/297 [00:00<?, ?B/s]

Precomputing [val]: 100%|██████████| 800/800 [03:50<00:00,  3.48it/s]

Precomputation complete.


In [ ]:
# dataloaders for 400-sample cache
import glob
from torch.utils.data import Dataset, DataLoader

BATCH_SIZE_400 = 16

class CachedASTDataset400(Dataset):
    """Loads precomputed AST feature tensors from disk."""
    def __init__(self, cache_dir, split):
        split_dir  = os.path.join(cache_dir, split)
        self.files = sorted(glob.glob(os.path.join(split_dir, "*.pt")))

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        item = torch.load(self.files[idx], weights_only=False)
        return item["input_values"], item["label"]

In [ ]:
train_dataset_400 = CachedASTDataset400(CACHE_400, "train")
val_dataset_400 = CachedASTDataset400(CACHE_400, "val")

train_loader_400 = DataLoader(train_dataset_400, batch_size=BATCH_SIZE_400,
                              shuffle=True,  num_workers=2, pin_memory=True)
val_loader_400 = DataLoader(val_dataset_400,   batch_size=BATCH_SIZE_400,
                              shuffle=False, num_workers=2, pin_memory=True)

print(f"Train: {len(train_dataset_400)} | Val: {len(val_dataset_400)}")

Train: 3200 | Val: 800


In [ ]:
# fresh model and optimiser — no weights carried over from the 200-sample run
import torch.nn as nn
import torch.optim as optim
from transformers import ASTForAudioClassification

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
LR_400 = 1e-5
WEIGHT_DECAY = 0.01
NUM_EPOCHS = 10

model_400 = ASTForAudioClassification.from_pretrained(
    AST_MODEL_ID, num_labels=10, ignore_mismatched_sizes=True
)
model_400 = model_400.to(DEVICE)

trainable = sum(p.numel() for p in model_400.parameters() if p.requires_grad)
print(f"Trainable parameters: {trainable:,}")

optimizer_400 = optim.AdamW(model_400.parameters(), lr=LR_400, weight_decay=WEIGHT_DECAY)
scheduler_400 = optim.lr_scheduler.CosineAnnealingLR(optimizer_400, T_max=NUM_EPOCHS)
criterion_400 = nn.CrossEntropyLoss()

Loading weights:   0%|          | 0/203 [00:00<?, ?it/s]

ASTForAudioClassification LOAD REPORT from: MIT/ast-finetuned-audioset-10-10-0.4593
Key                     | Status   |                                                                                        
------------------------+----------+----------------------------------------------------------------------------------------
classifier.dense.weight | MISMATCH | Reinit due to size mismatch ckpt: torch.Size([527, 768]) vs model:torch.Size([10, 768])
classifier.dense.bias   | MISMATCH | Reinit due to size mismatch ckpt: torch.Size([527]) vs model:torch.Size([10])          

Notes:
- MISMATCH	:ckpt weights were loaded, but they did not match the original empty weight shapes.


Trainable parameters: 86,196,490


In [ ]:
# training loop for 400-sample full fine-tune
from sklearn.metrics import f1_score, accuracy_score
from tqdm import tqdm

def run_epoch_400(model, loader, criterion, optimizer, training, device, desc=""):
    model.train() if training else model.eval()
    total_loss, preds, labels_all = 0.0, [], []

    ctx = torch.enable_grad() if training else torch.no_grad()
    with ctx:
        for inputs, labels in tqdm(loader, desc=desc):
            inputs = inputs.to(device)
            labels = labels.to(device)
            if training:
                optimizer.zero_grad()
            logits = model(inputs).logits
            loss = criterion(logits, labels)
            if training:
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                optimizer.step()
            total_loss += loss.item() * inputs.size(0)
            preds.extend(logits.argmax(dim=1).detach().cpu().numpy())
            labels_all.extend(labels.detach().cpu().numpy())

    avg_loss = total_loss / len(loader.dataset)
    f1 = f1_score(labels_all, preds, average="macro")
    acc = accuracy_score(labels_all, preds)
    return avg_loss, f1, acc

In [34]:
wandb.init(
    project="21f3002298-dl-genai-project",
    name="ast-src-full-finetune-400",
    config={
        "model":             AST_MODEL_ID,
        "epochs":            NUM_EPOCHS,
        "lr":                LR_400,
        "weight_decay":      WEIGHT_DECAY,
        "batch_size":        BATCH_SIZE_400,
        "frozen_layers":     "none (full fine-tune)",
        "samples_per_genre": SAMPLES_PER_GENRE,
        "data_seed":         DATA_SEED_400,
    }
)

In [ ]:
best_val_f1_400 = 0.0
MODEL_PATH_400 = os.path.join(DRIVE_ROOT, "models", "ast_src_full_finetune_400_best.pth")

for epoch in range(NUM_EPOCHS):
    train_loss, train_f1, train_acc = run_epoch_400(
        model_400, train_loader_400, criterion_400, optimizer_400, training=True,
        device=DEVICE, desc=f"Epoch {epoch+1}/{NUM_EPOCHS} [Train]"
    )
    val_loss, val_f1, val_acc = run_epoch_400(
        model_400, val_loader_400, criterion_400, optimizer_400, training=False,
        device=DEVICE, desc=f"Epoch {epoch+1}/{NUM_EPOCHS} [Val]"
    )
    scheduler_400.step()
    torch.cuda.empty_cache()

    wandb.log({
        "epoch":      epoch + 1,
        "train_loss": train_loss, "train_f1": train_f1, "train_acc": train_acc,
        "val_loss":   val_loss,   "val_f1":   val_f1,   "val_acc":   val_acc,
        "lr":         scheduler_400.get_last_lr()[0],
    })
    print(f"Epoch {epoch+1}/{NUM_EPOCHS}: train_f1={train_f1:.4f} | val_f1={val_f1:.4f}")

    if val_f1 > best_val_f1_400:
        best_val_f1_400 = val_f1
        torch.save(model_400.state_dict(), MODEL_PATH_400)
        print(f"  Saved best model (val_f1={val_f1:.4f})")

Epoch 1/10 [Val]: 100%|██████████| 50/50 [01:12<00:00,  1.44s/it]


Epoch 1/10: train_f1=0.7730 | val_f1=0.8972
  Saved best model (val_f1=0.8972)


Epoch 2/10 [Val]: 100%|██████████| 50/50 [01:11<00:00,  1.44s/it]


Epoch 2/10: train_f1=0.9387 | val_f1=0.9662
  Saved best model (val_f1=0.9662)


Epoch 3/10 [Val]: 100%|██████████| 50/50 [01:12<00:00,  1.45s/it]


Epoch 3/10: train_f1=0.9803 | val_f1=0.9638


... (10 outputs left out)

Epoch 9/10 [Val]: 100%|██████████| 50/50 [01:11<00:00,  1.43s/it]


Epoch 9/10: train_f1=1.0000 | val_f1=0.9750


Epoch 10/10 [Val]: 100%|██████████| 50/50 [01:12<00:00,  1.44s/it]


Epoch 10/10: train_f1=1.0000 | val_f1=0.9750


In [36]:
wandb.finish()

epoch,▁▂▃▃▄▅▆▆▇█
lr,█▇▇▆▅▃▂▂▁▁
train_acc,▁▆▇███████
train_f1,▁▆▇███████
train_loss,█▃▂▁▁▁▁▁▁▁
val_acc,▁▇▇▇▇█████
val_f1,▁▇▇▇▇█████
val_loss,█▂▃▃▂▁▁▁▁▁
epoch,10
lr,0
train_acc,1


## The End